# Fase 2 — Experiment Tracking con MLflow

## Objetivo de este notebook

En la Fase 1.3 comparamos 10 modelos de clasificación "a mano", guardando los resultados en un DataFrame dentro del notebook de EDA. Eso funciona para explorar, pero tiene un problema para lo que sigue del proyecto: si cerramos el notebook o lo volvemos a correr, perdemos el detalle de cada corrida (qué hiperparámetros se usaron, qué métricas dio exactamente, qué versión del modelo fue) — y no hay forma fácil de comparar experimentos entre sí ni de saber cuál modelo entrenado es "el bueno" para pasar a producción.

**MLflow** resuelve esto: es una herramienta de *experiment tracking* que registra automáticamente, para cada entrenamiento (cada "run"):
- los **parámetros** usados (hiperparámetros del modelo, semillas, etc.),
- las **métricas** obtenidas (recall, precision, f1, AUC-PR),
- **artefactos** (el modelo entrenado, gráficos, archivos),
- y metadatos (cuándo se corrió, con qué código).

Todo esto queda guardado localmente en una carpeta `mlruns/` (que no versionamos en git — se regenera al correr el código, igual que la data), y se puede explorar después con una interfaz visual (`mlflow ui`) para comparar corridas entre sí.

### Qué vamos a hacer en este notebook
1. Configurar MLflow y crear un *experimento* (un espacio con nombre donde se agrupan los runs relacionados).
2. Cargar los datos ya preparados (mismo split y preprocesamiento del EDA).
3. Entrenar y loguear a MLflow los modelos que ya identificamos como más prometedores en la Fase 1.3 (empezando por Hist Gradient Boosting, el elegido).
4. Comparar corridas desde la interfaz de MLflow.

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_sample_weight

import mlflow
import mlflow.sklearn

import optuna
from mlflow import MlflowClient

## Paso 1: Configurar MLflow y crear el experimento

Antes de entrenar nada, hay que decirle a MLflow dos cosas:

1. **Dónde guardar los runs** (`tracking_uri`): por defecto MLflow los guarda en una carpeta `mlruns/` en el directorio donde se ejecuta el código. Como este notebook vive en `notebooks/`, si no le decimos nada, la carpeta `mlruns/` quedaría *dentro* de `notebooks/` — para mantener la misma estructura ordenada del resto del proyecto (igual que `../data/raw/...`), la apuntamos explícitamente a la raíz del proyecto.
2. **En qué experimento agrupar los runs** (`experiment_name`): un experimento es como una carpeta con nombre que agrupa runs relacionados — todos los runs de comparación de modelos para este dataset van a ir bajo el mismo experimento, para poder compararlos entre sí en la interfaz de MLflow.

In [2]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("mantenimiento-predictivo-ai4i2020")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento activo:", mlflow.get_experiment_by_name("mantenimiento-predictivo-ai4i2020"))

2026/08/31 18:46:12 INFO mlflow.tracking.fluent: Experiment with name 'mantenimiento-predictivo-ai4i2020' does not exist. Creating a new experiment.


Tracking URI: file:../mlruns
Experimento activo: <Experiment: artifact_location='file:///c:/Users/marin/Desktop/APUN/aprendizaje-automatico-en-la-nube/notebooks/../mlruns/662614544400800058', creation_time=1788219972624, experiment_id='662614544400800058', last_update_time=1788219972624, lifecycle_stage='active', name='mantenimiento-predictivo-ai4i2020', tags={}>


**Análisis:** el `tracking_uri` quedó apuntando correctamente a `file:../mlruns` — como el notebook vive en `notebooks/`, esa ruta relativa sube un nivel y crea la carpeta `mlruns/` en la raíz del proyecto (se confirma en el `artifact_location` de la salida: `.../aprendizaje-automatico-en-la-nube/notebooks/../mlruns/...`, que resuelve a la raíz).

Como el experimento `mantenimiento-predictivo-ai4i2020` no existía todavía, MLflow lo creó automáticamente (se ve en el log `INFO mlflow.tracking.fluent: Experiment with name ... does not exist. Creating a new experiment`). De aquí en adelante, cualquier run que loguemos va a quedar agrupado bajo este mismo experimento, listo para comparar en la interfaz de MLflow.

## Paso 2: Cargar los datos y recrear el split

Para que este notebook sea autosuficiente (no dependa de tener abierto el notebook de EDA), repetimos aquí la carga del dataset y el mismo split que usamos en la Fase 1.3 — **mismo `random_state=42` y mismo `stratify=y`**, para que el train/test sea idéntico y los resultados sean comparables entre notebooks.

Recordemos: excluimos de `X` las 5 banderas de tipo de falla (TWF, HDF, PWF, OSF, RNF) porque serían "fugas de información" (data leakage) — esas banderas básicamente ya delatan la falla. También excluimos `UID` y `Product ID` por ser identificadores, no variables predictivas.

In [5]:
df = pd.read_csv("../data/raw/ai4i2020.csv", encoding="utf-8-sig")

X = df.drop(columns=["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Proporcion de fallas en train:", y_train.mean().round(4))
print("Proporcion de fallas en test:", y_test.mean().round(4))

X_train: (8000, 6) | X_test: (2000, 6)
Proporcion de fallas en train: 0.0339
Proporcion de fallas en test: 0.034


**Análisis:** el split quedó con 8.000 observaciones en train y 2.000 en test (80/20, como en la Fase 1.3), y las 6 columnas restantes en `X` (`Type` + las 5 variables numéricas) tras excluir identificadores y banderas de falla. La proporción de fallas se mantuvo prácticamente igual en ambos conjuntos (3.39% en train, 3.40% en test) gracias al `stratify=y` — esto confirma que el desbalance de clases no se distorsionó al partir los datos, así que la comparación de modelos que hagamos aquí será consistente con la de la Fase 1.3.

## Paso 3: Preprocesamiento

Igual que en la Fase 1.3, armamos un `ColumnTransformer` que:
- Escala las 5 variables numéricas con `StandardScaler` (media 0, desviación 1) — importante para modelos sensibles a la escala.
- Convierte `Type` (L/M/H) a variables dummy con `OneHotEncoder(drop="first")` — se elimina la primera categoría para evitar redundancia (multicolinealidad).

In [6]:
variables_numericas = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
variables_categoricas = ["Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), variables_numericas),
        ("cat", OneHotEncoder(drop="first"), variables_categoricas),
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Air temperature [K]',
                                  'Process temperature [K]',
                                  'Rotational speed [rpm]', 'Torque [Nm]',
                                  'Tool wear [min]']),
                                ('cat', OneHotEncoder(drop='first'), ['Type'])])


**Análisis:** el `ColumnTransformer` quedó configurado igual que en la Fase 1.3 — escala las 5 variables numéricas y convierte `Type` en variables dummy (elimina la categoría base para evitar redundancia). Con esto ya tenemos todo listo para entrenar y empezar a trackear con MLflow.

## Paso 4: Entrenar el modelo elegido (Hist Gradient Boosting) con tracking en MLflow

En la Fase 1.3 identificamos **Hist Gradient Boosting** como el modelo más robusto entre los 10 comparados. Ahora lo entrenamos de nuevo, pero esta vez envolviendo todo dentro de un **run de MLflow** (`mlflow.start_run()`), para que quede registrado automáticamente:

- **Parámetros** (`mlflow.log_param`): qué modelo es, cómo se manejó el desbalance de clases, la semilla usada.
- **Métricas** (`mlflow.log_metric`): Recall, Precision, F1 y AUC-PR de la clase "falla" — las mismas que usamos en la Fase 1.3, por consistencia.
- **El modelo entrenado como artefacto** (`mlflow.sklearn.log_model`): queda guardado el pipeline completo (preprocesamiento + modelo), listo para poder cargarlo después sin reentrenar.

Mantenemos el mismo manejo de desbalance que ya justificamos en la Fase 1.3: como `HistGradientBoostingClassifier` no soporta `class_weight` nativo, usamos `sample_weight` calculado con `compute_sample_weight`.

In [7]:
with mlflow.start_run(run_name="hist_gradient_boosting_baseline"):
    modelo = HistGradientBoostingClassifier(random_state=42)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weight_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_proba)

    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("recall_falla", recall)
    mlflow.log_metric("precision_falla", precision)
    mlflow.log_metric("f1_falla", f1)
    mlflow.log_metric("auc_pr", auc_pr)

    mlflow.sklearn.log_model(pipeline, "modelo")

    print("Run ID:", mlflow.active_run().info.run_id)
    print(f"Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f} | AUC-PR: {auc_pr:.4f}")

2026/08/31 18:51:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/31 18:51:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: f57af4cd3b7c43d3b8d4bf38e96661fb
Recall: 0.7941 | Precision: 0.7297 | F1: 0.7606 | AUC-PR: 0.8343


**Análisis:** el primer run quedó registrado en MLflow (`Run ID: f57af4cd3b7c43d3b8d4bf38e96661fb`) con Recall 0.7941, Precision 0.7297, F1 0.7606 y AUC-PR 0.8343.

Comparando con la meta declarada en la Sección 1.1 (Recall ≥ 0.80 en la clase de falla), quedamos **muy cerca pero por debajo** (0.7941 vs. 0.80). Esto no es un problema — es justo la motivación para las tareas pendientes de esta fase: ajustar el umbral de decisión (por defecto el modelo predice con corte en 0.5, pero podemos bajarlo para priorizar más el Recall a costa de algo de Precision), tunear hiperparámetros con Optuna, y probar SMOTE. Ahora que tenemos este primer run como punto de referencia ("baseline") registrado en MLflow, cualquier mejora que hagamos después la podemos comparar objetivamente contra este número.

**Análisis:** confirmado visualmente en la interfaz de MLflow — el run `hist_gradient_boosting_baseline` quedó registrado dentro del experimento `mantenimiento-predictivo-ai4i2020`, con el modelo (pipeline completo de preprocesamiento + clasificador) guardado como artefacto. A partir de aquí, cada vez que entrenemos una variante (otro modelo, otros hiperparámetros, con SMOTE, etc.) va a aparecer como una fila más en esta misma tabla, lo que nos permite comparar experimentos de forma ordenada y objetiva — justo el problema que teníamos en la Fase 1.3 al comparar los 10 modelos "a mano" en un DataFrame que se perdía al cerrar el notebook.

## Paso 5: Validar con cross-validation

Hasta ahora evaluamos el modelo con un único train/test split (80/20) — eso da una sola medición que puede depender un poco de qué observaciones cayeron por azar en el test. La validación cruzada entrena y evalúa el modelo varias veces con distintas particiones, dándonos un promedio y una desviación estándar por métrica: una estimación más robusta de qué tan bien generaliza el modelo.

Usamos `StratifiedKFold` con 5 folds (mantiene la proporción de fallas en cada partición) sobre `X_train`/`y_train`, y logueamos el resultado como un nuevo run en MLflow, para poder compararlo contra el run del split simple.

In [10]:
with mlflow.start_run(run_name="hist_gradient_boosting_cv5"):
    modelo = HistGradientBoostingClassifier(random_state=42)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    resultados_cv = cross_validate(
    pipeline, X_train, y_train,
    cv=cv,
    scoring=["recall", "precision", "f1"],
    params={"classifier__sample_weight": sample_weight_train},
)

    recall_mean, recall_std = resultados_cv["test_recall"].mean(), resultados_cv["test_recall"].std()
    precision_mean, precision_std = resultados_cv["test_precision"].mean(), resultados_cv["test_precision"].std()
    f1_mean, f1_std = resultados_cv["test_f1"].mean(), resultados_cv["test_f1"].std()

    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("recall_falla_mean", recall_mean)
    mlflow.log_metric("recall_falla_std", recall_std)
    mlflow.log_metric("precision_falla_mean", precision_mean)
    mlflow.log_metric("precision_falla_std", precision_std)
    mlflow.log_metric("f1_falla_mean", f1_mean)
    mlflow.log_metric("f1_falla_std", f1_std)

    print(f"Recall: {recall_mean:.4f} +/- {recall_std:.4f}")
    print(f"Precision: {precision_mean:.4f} +/- {precision_std:.4f}")
    print(f"F1: {f1_mean:.4f} +/- {f1_std:.4f}")

Recall: 0.7712 +/- 0.0449
Precision: 0.6710 +/- 0.0557
F1: 0.7164 +/- 0.0433


**Análisis:** con cross-validation de 5 folds, Hist Gradient Boosting da Recall 0.7712 (± 0.0449), Precision 0.6710 (± 0.0557) y F1 0.7164 (± 0.0433).

Comparado con el run del split simple (Recall 0.7941, Precision 0.7297, F1 0.7606), el promedio de cross-validation queda un poco más bajo en las tres métricas. Esto es esperable y es justo el valor de hacer cross-validation: el split original nos dio una medición algo optimista por cómo cayeron los datos en ese test particular, mientras que el promedio de 5 folds es una estimación más realista y estable de cómo se comportaría el modelo con datos nuevos. La desviación estándar (±0.04-0.06) también nos dice que hay variabilidad razonable entre folds — no es un modelo perfectamente estable, algo a tener en cuenta antes de prometer un Recall exacto en producción.

## Paso 6: Loguear un segundo modelo para comparar (Random Forest)

Random Forest fue otro de los modelos con buen desempeño en la comparación de 10 modelos de la Fase 1.3. A diferencia de Hist Gradient Boosting, Random Forest **sí soporta `class_weight="balanced"` nativamente** (no necesita `sample_weight` por separado). Lo entrenamos igual, con cross-validation de 5 folds, y lo logueamos como otro run dentro del mismo experimento para poder comparar ambos modelos lado a lado en la interfaz de MLflow.

In [12]:
with mlflow.start_run(run_name="random_forest_cv5"):
    modelo = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    resultados_cv = cross_validate(
        pipeline, X_train, y_train,
        cv=cv,
        scoring=["recall", "precision", "f1"],
    )

    recall_mean, recall_std = resultados_cv["test_recall"].mean(), resultados_cv["test_recall"].std()
    precision_mean, precision_std = resultados_cv["test_precision"].mean(), resultados_cv["test_precision"].std()
    f1_mean, f1_std = resultados_cv["test_f1"].mean(), resultados_cv["test_f1"].std()

    mlflow.log_param("modelo", "RandomForestClassifier")
    mlflow.log_param("manejo_desbalance", "class_weight balanced")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("recall_falla_mean", recall_mean)
    mlflow.log_metric("recall_falla_std", recall_std)
    mlflow.log_metric("precision_falla_mean", precision_mean)
    mlflow.log_metric("precision_falla_std", precision_std)
    mlflow.log_metric("f1_falla_mean", f1_mean)
    mlflow.log_metric("f1_falla_std", f1_std)

    print(f"Recall: {recall_mean:.4f} +/- {recall_std:.4f}")
    print(f"Precision: {precision_mean:.4f} +/- {precision_std:.4f}")
    print(f"F1: {f1_mean:.4f} +/- {f1_std:.4f}")

Recall: 0.6643 +/- 0.0196
Precision: 0.7282 +/- 0.0624
F1: 0.6934 +/- 0.0313


**Análisis:** con cross-validation, Random Forest da Recall 0.6643 (± 0.0196), Precision 0.7282 (± 0.0624) y F1 0.6934 (± 0.0313).

Comparando ambos modelos con la misma metodología (CV de 5 folds):

| Modelo | Recall | Precision | F1 |
|---|---|---|---|
| Hist Gradient Boosting | 0.7712 ± 0.045 | 0.6710 ± 0.056 | 0.7164 ± 0.043 |
| Random Forest | 0.6643 ± 0.020 | 0.7282 ± 0.062 | 0.6934 ± 0.031 |

Random Forest tiene mejor Precision, pero **peor Recall** — y recordemos que Recall es la métrica prioritaria según la Sección 1.1 (no detectar una falla real es el error más caro). Esto confirma, ahora con cross-validation y no solo con un split, la elección que ya habíamos hecho en la Fase 1.3: **Hist Gradient Boosting sigue siendo el modelo más adecuado** para este problema, porque prioriza mejor el tipo de error que más nos interesa evitar.

## Paso 7: Ajustar el umbral de decisión

Cuando el modelo predice con `.predict()`, por dentro usa un umbral de 0.5 sobre la probabilidad: si P(falla) > 0.5, predice falla. Ese 0.5 es arbitrario, es solo el default de sklearn — no viene de ninguna consideración de negocio.

Como priorizamos Recall (Sección 1.1), podemos **bajar el umbral**: así el modelo "dispara la alarma" más fácilmente, capturando más fallas reales (mejor Recall) a costa de más falsas alarmas (peor Precision). Probamos varios umbrales sobre las probabilidades predichas en el test, buscando el punto donde el Recall se acerque a la meta de 0.80 sin sacrificar demasiada Precision.

In [13]:
modelo_final = HistGradientBoostingClassifier(random_state=42)
pipeline_final = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo_final)])

sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
pipeline_final.fit(X_train, y_train, classifier__sample_weight=sample_weight_train)

y_proba = pipeline_final.predict_proba(X_test)[:, 1]

for umbral in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]:
    y_pred_umbral = (y_proba >= umbral).astype(int)
    recall = recall_score(y_test, y_pred_umbral)
    precision = precision_score(y_test, y_pred_umbral, zero_division=0)
    f1 = f1_score(y_test, y_pred_umbral)
    print(f"Umbral {umbral:.2f} -> Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f}")

Umbral 0.50 -> Recall: 0.7941 | Precision: 0.7297 | F1: 0.7606
Umbral 0.40 -> Recall: 0.7941 | Precision: 0.6835 | F1: 0.7347
Umbral 0.30 -> Recall: 0.8088 | Precision: 0.6044 | F1: 0.6918
Umbral 0.25 -> Recall: 0.8235 | Precision: 0.5833 | F1: 0.6829
Umbral 0.20 -> Recall: 0.8382 | Precision: 0.5327 | F1: 0.6514
Umbral 0.15 -> Recall: 0.8529 | Precision: 0.4793 | F1: 0.6138
Umbral 0.10 -> Recall: 0.8824 | Precision: 0.4444 | F1: 0.5911


**Análisis:** bajando el umbral de 0.5 a **0.30**, el Recall pasa de 0.7941 a 0.8088 — ¡ya alcanzamos la meta de la Sección 1.1 (Recall ≥ 0.80)! El costo es la Precision, que baja de 0.7297 a 0.6044 (más falsas alarmas). Si seguimos bajando el umbral, el Recall sube más, pero la Precision se desploma (a umbral 0.10, Precision cae a 0.4444 — casi la mitad de las alarmas serían falsas).

Elegimos **umbral = 0.30** como punto de operación: es el primer umbral que cumple la meta de Recall sin sacrificar la Precision más de lo necesario (se mantiene por encima de 0.60 — 6 de cada 10 alarmas son fallas reales). Esto ilustra bien que la elección del umbral es una decisión de negocio, no solo técnica: el punto óptimo depende de cuánto cuesta en la planta real cada tipo de error (una parada no planeada vs. una revisión de mantenimiento innecesaria).

In [14]:
with mlflow.start_run(run_name="hist_gradient_boosting_umbral_030"):
    umbral_elegido = 0.30
    y_pred_umbral = (y_proba >= umbral_elegido).astype(int)

    recall = recall_score(y_test, y_pred_umbral)
    precision = precision_score(y_test, y_pred_umbral, zero_division=0)
    f1 = f1_score(y_test, y_pred_umbral)
    auc_pr = average_precision_score(y_test, y_proba)

    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("umbral_decision", umbral_elegido)

    mlflow.log_metric("recall_falla", recall)
    mlflow.log_metric("precision_falla", precision)
    mlflow.log_metric("f1_falla", f1)
    mlflow.log_metric("auc_pr", auc_pr)

    mlflow.sklearn.log_model(pipeline_final, "modelo")

    print(f"Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f} | AUC-PR: {auc_pr:.4f}")

2026/08/31 19:10:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/31 19:10:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Recall: 0.8088 | Precision: 0.6044 | F1: 0.6918 | AUC-PR: 0.8343


## Paso 8: Tuning de hiperparámetros con Optuna

Hasta ahora usamos Hist Gradient Boosting con sus hiperparámetros por defecto. Optuna busca automáticamente, dentro de rangos que le definimos, la combinación de hiperparámetros que maximiza una métrica objetivo — de forma más eficiente que probar a mano o con grid search exhaustivo (usa un algoritmo bayesiano que aprende de los intentos anteriores para decidir qué probar después).

Como métrica objetivo usamos **AUC-PR** y no Recall directamente: AUC-PR resume qué tan bien el modelo separa las dos clases en general, sin depender de un umbral fijo — así evitamos que Optuna "haga trampa" ajustando hiperparámetros para maximizar Recall a costa de un modelo degenerado (por ejemplo, uno que casi siempre prediga "falla"). Una vez tengamos los mejores hiperparámetros, podemos volver a aplicar el ajuste de umbral del Paso 7 sobre el modelo ya optimizado.

Buscamos sobre `max_iter`, `max_depth`, `learning_rate`, `max_leaf_nodes` y `l2_regularization` (los hiperparámetros más influyentes de este modelo), con cross-validation de 3 folds por intento (para no demorarnos demasiado) y 20 intentos (`n_trials`) en total.

In [15]:
def objetivo(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 60),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 1.0),
    }

    modelo = HistGradientBoostingClassifier(random_state=42, **params)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=cv,
        scoring="average_precision",
        params={"classifier__sample_weight": sample_weight_train},
    )
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objetivo, n_trials=20)

print("Mejor AUC-PR (CV):", study.best_value)
print("Mejores hiperparametros:", study.best_params)

[I 2026-08-31 19:11:20,846] A new study created in memory with name: no-name-3a3212db-c0cb-429c-a4b8-3f83186155c9
[I 2026-08-31 19:11:22,994] Trial 0 finished with value: 0.7965583383644778 and parameters: {'max_iter': 95, 'max_depth': 12, 'learning_rate': 0.062450786943598464, 'max_leaf_nodes': 47, 'l2_regularization': 0.04057281949123748}. Best is trial 0 with value: 0.7965583383644778.
[I 2026-08-31 19:11:24,968] Trial 1 finished with value: 0.7801313961091346 and parameters: {'max_iter': 167, 'max_depth': 5, 'learning_rate': 0.22532934625525297, 'max_leaf_nodes': 18, 'l2_regularization': 0.660346461750303}. Best is trial 0 with value: 0.7965583383644778.
[I 2026-08-31 19:11:27,652] Trial 2 finished with value: 0.7962067955419196 and parameters: {'max_iter': 154, 'max_depth': 9, 'learning_rate': 0.08158495567125565, 'max_leaf_nodes': 56, 'l2_regularization': 0.7184745979967636}. Best is trial 0 with value: 0.7965583383644778.
[I 2026-08-31 19:11:29,701] Trial 3 finished with value: 

Mejor AUC-PR (CV): 0.7997757504722348
Mejores hiperparametros: {'max_iter': 223, 'max_depth': 8, 'learning_rate': 0.057075747161782854, 'max_leaf_nodes': 54, 'l2_regularization': 0.21725275745972278}


**Análisis:** Optuna encontró hiperparámetros con AUC-PR promedio (CV de 3 folds) de 0.7998 — similar al 0.8343 que obtuvimos con hiperparámetros por defecto (no son directamente comparables número a número, porque uno es CV sobre train y el otro es sobre el test, pero da una señal de que el modelo por defecto ya estaba bastante bien calibrado; el tuning aporta una mejora marginal más que un salto grande, algo común en datasets donde el modelo base ya captura bien la señal).

Los mejores hiperparámetros: `max_iter=223`, `max_depth=8`, `learning_rate≈0.057`, `max_leaf_nodes=54` — en general, más iteraciones y árboles un poco más profundos que el default, con un aprendizaje más gradual (learning rate más bajo), lo típico de modelos que generalizan mejor en boosting.

## Paso 9: Modelo final (hiperparámetros optimizados + umbral ajustado)

Combinamos los dos hallazgos de esta fase: los mejores hiperparámetros de Optuna y el umbral de decisión de 0.30. Entrenamos el modelo final con `study.best_params`, evaluamos en el test set con ese umbral, y logueamos el resultado como el run "candidato final" de esta fase.

In [16]:
with mlflow.start_run(run_name="hist_gradient_boosting_optuna_umbral_030"):
    mejores_params = study.best_params
    modelo_final = HistGradientBoostingClassifier(random_state=42, **mejores_params)
    pipeline_final = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", modelo_final)])

    sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
    pipeline_final.fit(X_train, y_train, classifier__sample_weight=sample_weight_train)

    y_proba = pipeline_final.predict_proba(X_test)[:, 1]

    umbral_elegido = 0.30
    y_pred_umbral = (y_proba >= umbral_elegido).astype(int)

    recall = recall_score(y_test, y_pred_umbral)
    precision = precision_score(y_test, y_pred_umbral, zero_division=0)
    f1 = f1_score(y_test, y_pred_umbral)
    auc_pr = average_precision_score(y_test, y_proba)

    mlflow.log_params(mejores_params)
    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "sample_weight (balanced)")
    mlflow.log_param("umbral_decision", umbral_elegido)
    mlflow.log_param("tuning", "optuna_20_trials")

    mlflow.log_metric("recall_falla", recall)
    mlflow.log_metric("precision_falla", precision)
    mlflow.log_metric("f1_falla", f1)
    mlflow.log_metric("auc_pr", auc_pr)

    mlflow.sklearn.log_model(pipeline_final, "modelo")

    print(f"Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f} | AUC-PR: {auc_pr:.4f}")

2026/08/31 19:14:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/31 19:14:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Recall: 0.8235 | Precision: 0.6022 | F1: 0.6957 | AUC-PR: 0.8485


**Análisis:** el modelo final (hiperparámetros de Optuna + umbral 0.30) da Recall 0.8235, Precision 0.6022, F1 0.6957 y AUC-PR 0.8485 — mejora tanto el Recall (0.8235 vs. 0.8088 con hiperparámetros por defecto) como el AUC-PR (0.8485 vs. 0.8343), con Precision prácticamente igual. Es una mejora modesta pero real: confirma que vale la pena el tuning, aunque el modelo por defecto ya estaba razonablemente bien calibrado. Con esto superamos con margen la meta de Recall ≥ 0.80 de la Sección 1.1, manteniendo una Precision aceptable (6 de cada 10 alarmas son fallas reales).

## Paso 10: Probar SMOTE como alternativa al manejo de desbalance

Hasta ahora manejamos el desbalance de clases con pesos (`sample_weight`), que le dicen al modelo "presta más atención a los errores en la clase minoritaria" sin tocar los datos. **SMOTE** (Synthetic Minority Oversampling Technique) es distinto: genera observaciones sintéticas de la clase minoritaria (falla) interpolando entre vecinos cercanos reales, para balancear el dataset de entrenamiento antes de entrenar.

Detalle importante: SMOTE **solo se debe aplicar sobre los datos de entrenamiento**, nunca sobre el test — si no, estaríamos filtrando información sintética hacia la evaluación e inflando artificialmente los resultados. Para que esto se respete automáticamente dentro de cross-validation (donde en cada fold cambia qué es "train"), no podemos usar el `Pipeline` normal de sklearn — usamos el `Pipeline` de `imbalanced-learn`, que aplica SMOTE solo dentro del fold de entrenamiento de cada iteración. Como SMOTE ya balancea las clases por su cuenta, aquí no usamos `sample_weight` (sería redundante).

In [18]:
with mlflow.start_run(run_name="hist_gradient_boosting_smote_cv5"):
    pipeline_smote = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", HistGradientBoostingClassifier(random_state=42, **study.best_params)),
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    resultados_cv_smote = cross_validate(
        pipeline_smote, X_train, y_train,
        cv=cv,
        scoring=["recall", "precision", "f1", "average_precision"],
    )

    recall_mean = resultados_cv_smote["test_recall"].mean()
    precision_mean = resultados_cv_smote["test_precision"].mean()
    f1_mean = resultados_cv_smote["test_f1"].mean()
    auc_pr_mean = resultados_cv_smote["test_average_precision"].mean()

    mlflow.log_params(study.best_params)
    mlflow.log_param("modelo", "HistGradientBoostingClassifier")
    mlflow.log_param("manejo_desbalance", "SMOTE")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("recall_falla_mean", recall_mean)
    mlflow.log_metric("precision_falla_mean", precision_mean)
    mlflow.log_metric("f1_falla_mean", f1_mean)
    mlflow.log_metric("auc_pr_mean", auc_pr_mean)

    print(f"Recall: {recall_mean:.4f} | Precision: {precision_mean:.4f} | F1: {f1_mean:.4f} | AUC-PR: {auc_pr_mean:.4f}")

Recall: 0.7935 | Precision: 0.5726 | F1: 0.6642 | AUC-PR: 0.7894


**Análisis:** con SMOTE (mismos hiperparámetros optimizados de Optuna), a umbral por defecto (0.5), obtenemos Recall 0.7935, Precision 0.5726, F1 0.6642, AUC-PR 0.7894.

Comparado con `sample_weight` (Paso 5, mismo modelo pero con hiperparámetros por defecto): Recall 0.7712, Precision 0.6710. SMOTE sube un poco el Recall pero baja bastante más la Precision, y el AUC-PR (0.7894) queda por debajo del que veníamos logrando con sample_weight + tuning (0.8485). Esto sugiere que, para este dataset, generar observaciones sintéticas no aporta tanto como simplemente ponderar los errores — el desbalance (3.39%) no es tan extremo como para que la escasez de ejemplos reales sea el cuello de botella; el modelo ya tiene suficientes casos reales de falla para aprender el patrón sin necesitar datos sintéticos.

**Conclusión de la Fase 2:** nos quedamos con **Hist Gradient Boosting + hiperparámetros de Optuna + sample_weight (balanced) + umbral de decisión 0.30** como modelo candidato final — es la combinación con mejor Recall y AUC-PR de todas las probadas (Recall 0.8235, Precision 0.6022, AUC-PR 0.8485), superando la meta de Recall ≥ 0.80 de la Sección 1.1. SMOTE quedó descartado como alternativa para este dataset en particular.

## Paso 11: Registrar el modelo ganador en el Model Registry

El Model Registry de MLflow es donde se versionan los modelos "candidatos a producción" — cada vez que registras un modelo bajo un nombre, MLflow le asigna una versión (v1, v2, ...) y le puedes poner una etiqueta (alias) de qué tan probado está (por ejemplo, `champion` para "el mejor hasta ahora"). Es la forma en que un equipo se comunica "este es el modelo bueno" sin mandarse archivos sueltos.

Buscamos el run del Paso 9 (Hist Gradient Boosting + Optuna + umbral 0.30 — nuestro modelo candidato final) por su nombre, lo registramos bajo `mantenimiento-predictivo-hgb`, y le asignamos el alias `champion`.

In [20]:
runs = mlflow.search_runs(
    experiment_names=["mantenimiento-predictivo-ai4i2020"],
    filter_string="tags.mlflow.runName = 'hist_gradient_boosting_optuna_umbral_030'",
    order_by=["start_time DESC"],
)

run_id = runs.iloc[0]["run_id"]
print("Run ID encontrado:", run_id)

model_uri = f"runs:/{run_id}/modelo"
resultado_registro = mlflow.register_model(model_uri=model_uri, name="mantenimiento-predictivo-hgb")

print("Modelo registrado:", resultado_registro.name, "- version", resultado_registro.version)

client = MlflowClient()
client.set_registered_model_alias(
    name="mantenimiento-predictivo-hgb",
    alias="champion",
    version=resultado_registro.version,
)

print(f"Alias 'champion' asignado a la version {resultado_registro.version}")

Run ID encontrado: a589df2b7363466c8937697632a614fb
Modelo registrado: mantenimiento-predictivo-hgb - version 1
Alias 'champion' asignado a la version 1


Successfully registered model 'mantenimiento-predictivo-hgb'.
Created version '1' of model 'mantenimiento-predictivo-hgb'.


**Análisis:** el modelo quedó registrado como `mantenimiento-predictivo-hgb`, versión 1, con el alias `champion` — confirmado tanto por la salida (`Successfully registered model...`) como por los prints. A partir de ahora, cualquier código (por ejemplo, la futura API de FastAPI en la Fase 4) puede cargar "el modelo campeón actual" simplemente pidiendo `models:/mantenimiento-predictivo-hgb@champion`, sin necesitar saber el run_id exacto ni volver a entrenar nada. Si en el futuro entrenamos una versión mejor, se registra como versión 2 y se le puede mover el alias `champion` — quedando trazabilidad completa de cuál era el modelo en producción en cada momento.

## Conclusiones generales — Fase 2

- MLflow quedó configurado y funcionando de punta a punta: tracking de parámetros, métricas y modelos, con 6 runs logueados en el experimento `mantenimiento-predictivo-ai4i2020`.
- Se confirmó, ahora con cross-validation, la elección de Hist Gradient Boosting sobre Random Forest hecha en la Fase 1.3.
- Ajustar el umbral de decisión a 0.30 (en vez del 0.5 por defecto) permitió alcanzar la meta de Recall ≥ 0.80 de la Sección 1.1.
- El tuning con Optuna aportó una mejora adicional, aunque modesta, sobre los hiperparámetros por defecto.
- SMOTE no superó al manejo de desbalance por pesos (`sample_weight`) para este dataset — se descarta como técnica para este caso particular.
- **Modelo candidato final:** Hist Gradient Boosting + hiperparámetros de Optuna + `sample_weight` balanceado + umbral 0.30 (Recall 0.8235, Precision 0.6022, AUC-PR 0.8485), registrado en el Model Registry como `mantenimiento-predictivo-hgb` con alias `champion`.
- **Pendiente para una próxima sesión:** documentar todo esto en `docs/03_resultados_experiment_tracking.md` (mismo patrón que `docs/02_resultados_eda.md`), y actualizar el README con el estado de la Fase 2.

In [5]:
import pandas as pd

df = pd.read_csv("../data/raw/ai4i2020.csv", encoding="utf-8-sig")
print(df.describe())
print(df["Type"].unique())

               UDI  Air temperature [K]  Process temperature [K]  \
count  10000.00000         10000.000000             10000.000000   
mean    5000.50000           300.004930               310.005560   
std     2886.89568             2.000259                 1.483734   
min        1.00000           295.300000               305.700000   
25%     2500.75000           298.300000               308.800000   
50%     5000.50000           300.100000               310.100000   
75%     7500.25000           301.500000               311.100000   
max    10000.00000           304.500000               313.800000   

       Rotational speed [rpm]   Torque [Nm]  Tool wear [min]  Machine failure  \
count            10000.000000  10000.000000     10000.000000     10000.000000   
mean              1538.776100     39.986910       107.951000         0.033900   
std                179.284096      9.968934        63.654147         0.180981   
min               1168.000000      3.800000         0.000000   